# Migrate the CSVs to the postgresql database

In [7]:
import os
import psycopg2
# import pandas as pd
# import numpy as np
# from datetime import datetime, timedelta
from dotenv import load_dotenv
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

In [8]:
load_dotenv()

DB_NAME = os.getenv("POSTGRES_DB")
DB_USER = os.getenv("POSTGRES_USER")
DB_PASSWORD = os.getenv("POSTGRES_PASSWORD")
DB_HOST = "localhost"
DB_PORT = 5432

# Create Database

In [9]:
def create_database():
    """
    Creates a new database if it doesn't exist yet.
    """
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
    cursor = conn.cursor()

    cursor.execute(f"SELECT 1 FROM pg_database WHERE datname = '{DB_NAME}';")
    exists = cursor.fetchone()

    if not exists:
        cursor.execute(f"CREATE DATABASE {DB_NAME};")
        print(f"Database '{DB_NAME}' created successfully.")
    else:
        print(f"Database '{DB_NAME}' already exists.")

In [10]:
create_database()

Database 'postgres' already exists.


# Set up the tables

In [11]:
def setup_tables():
    """
    Creates simple tables for stocks and daily OHLCV.
    """
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS companies (
            id SERIAL,
            ticker TEXT PRIMARY KEY,
            company_name TEXT,
            market_cap NUMERIC,
            country TEXT
        );
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS ohlcv (
            ticker TEXT NOT NULL REFERENCES companies(ticker),
            timestamp TIMESTAMPTZ NOT NULL,
            open DOUBLE PRECISION,
            high DOUBLE PRECISION,
            low DOUBLE PRECISION,
            close DOUBLE PRECISION,
            volume BIGINT,
            dividends REAL,
            stock_splits REAL,
            PRIMARY KEY (ticker, timestamp)
        );

        SELECT create_hypertable(
            'ohlcv',
            'timestamp',
            chunk_time_interval => interval '7 days',
            partitioning_column => 'ticker',
            number_partitions => 2000
        );
    """)

    conn.commit()
    cursor.close()
    conn.close()
    print("Tables created.")

In [12]:
setup_tables()

Tables created.
